In [0]:
# Delta Time Travel & Audit Demo (Comprehensive Architecture Showcase)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

CATALOG = "ecomstore.ecomlake"

# CDC UPSERTS (Tracking State Changes without Duplication)
# Tables: Orders, Payments, Shipments

print("CDC Upserts (Orders, Payments, Shipments)")

def show_cdc_metrics(table_name, entity_id_col, status_col):
    full_table = f"{CATALOG}.{table_name}"
    print(f"\n---> Auditing: {full_table} <---")
    
    print("📈 Proof of CDC (Merge Metrics):")
    display(spark.sql(f"""
        SELECT 
            version, timestamp, operation, 
            operationMetrics.numTargetRowsInserted AS rows_inserted,
            operationMetrics.numTargetRowsUpdated AS rows_updated
        FROM (DESCRIBE HISTORY {full_table})
        WHERE operation = 'MERGE'
        ORDER BY version DESC
        LIMIT 2
    """))

    print(f"🔄 Entities that changed {status_col} between Version 0 and Latest:")
    try:
        v0_df = spark.read.format("delta").option("versionAsOf", 0).table(full_table).select(entity_id_col, col(status_col).alias("status_v0"))
        v_latest_df = spark.read.table(full_table).select(entity_id_col, col(status_col).alias("status_latest"))

        (
            display(v0_df.join(v_latest_df, on=entity_id_col)
            .filter(col("status_v0") != col("status_latest"))
            .groupBy("status_v0", "status_latest")
            .count())
        )
    except Exception as e:
        print(f"Not enough history yet to compare Version 0 with Latest. Run more batches!\n")

# Execute CDC Checks
show_cdc_metrics("silver_orders", "order_id", "status")
show_cdc_metrics("silver_payments", "payment_id", "payment_status")
show_cdc_metrics("silver_shipments", "shipment_id", "shipment_status")

# SCD TYPE 1 (Overwrites & Data Recovery)
# Tables: Products
# Proving that Delta Lake remembers data even after we explicitly overwrite it!

print("Recovering Overwritten Data (SCD Type 1 - Products)")

products_table = f"{CATALOG}.silver_products"

print("\n🔍 Checking Price Changes (Version 0 vs Latest):")
try:
    v0_products = spark.read.format("delta").option("versionAsOf", 0).table(products_table).select("product_id", col("selling_price").alias("price_v0"))
    v_latest_products = spark.read.table(products_table).select("product_id", col("selling_price").alias("price_latest"))

    (
        display(v0_products.join(v_latest_products, on="product_id")
        .filter(col("price_v0") != col("price_latest"))
        .select("product_id", "price_v0", "price_latest")
        .limit(5))
    )
    print("Even though SCD Type 1 explicitly overwrites the price, Delta Time Travel lets us audit the old price!")
except Exception as e:
    print("Not enough history yet to compare Version 0 with Latest.")

# SCD TYPE 2 (Delta Log vs Business Logic)
# Tables: Customers, Sellers

print("Delta Log vs Business Logic (SCD Type 2 - Customers & Sellers)")

def show_scd2_history(table_name):
    full_table = f"{CATALOG}.{table_name}"
    print(f"\n📋 Delta Transaction History for: {full_table}")
    display(spark.sql(f"""
        SELECT version, timestamp, operation 
        FROM (DESCRIBE HISTORY {full_table})
        ORDER BY version DESC
        LIMIT 3
    """))

show_scd2_history("silver_customers_scd")
show_scd2_history("silver_sellers_scd")

print("For Customers and Sellers, we use SCD Type 2. The business history is stored IN the table columns (start_date/end_date).")
print("Therefore, we don't use Delta Time Travel to find old profiles; we use it strictly to audit WHEN the pipeline ran.")

# System Governance & Rollbacks

print(" Governance & Disaster Recovery")

orders_table = f"{CATALOG}.silver_orders"
print("To configure a table to retain history for 30 days:")
print(f"ALTER TABLE {orders_table} SET TBLPROPERTIES ('delta.logRetentionDuration' = 'interval 30 days');")

print("\nTo permanently delete old files and save storage costs (Do not run during demo):")
print(f"VACUUM {orders_table} RETAIN 168 HOURS;")

print("\nTo rescue a table if a bad script accidentally deletes everything:")
print(f"RESTORE TABLE {orders_table} TO VERSION AS OF 0;")